In [1]:
import os, random, numpy as np, cv2
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torchvision.transforms as T
from pycocotools.coco import COCO

# Segmentation models
!pip install -q segmentation-models-pytorch
import segmentation_models_pytorch as smp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 69.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour i

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
data_dir = "/kaggle/input/coco-2017-dataset/coco2017"
img_dir = os.path.join(data_dir, "train2017")
ann_file = os.path.join(data_dir, "annotations", "instances_train2017.json")

coco = COCO(ann_file)
img_ids = coco.getImgIds()
random_ids = random.sample(img_ids, 5)  # or 5 for more samples
images = coco.loadImgs(random_ids)

loading annotations into memory...
Done (t=21.06s)
creating index...
index created!


In [3]:
def load_image_and_mask(img_meta):
    img_path = os.path.join(img_dir, img_meta['file_name'])
    image = Image.open(img_path).convert("RGB")
    image = image.resize((256, 256))
    img_tensor = T.ToTensor()(image)

    ann_ids = coco.getAnnIds(imgIds=img_meta['id'], iscrowd=None)
    anns = coco.loadAnns(ann_ids)

    mask = np.zeros((img_meta['height'], img_meta['width']), dtype=np.uint8)
    for ann in anns:
        mask = np.maximum(mask, coco.annToMask(ann) * ann['category_id'])
    mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)
    mask_tensor = torch.tensor(mask, dtype=torch.long)

    return img_tensor, mask_tensor

samples = [load_image_and_mask(img) for img in images]

In [4]:
def train_model(encoder_name, samples, epochs=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = smp.Unet(encoder_name=encoder_name, encoder_weights="imagenet", classes=91, activation=None)
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = torch.nn.CrossEntropyLoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for img, mask in samples:
            img = img.unsqueeze(0).to(device)
            mask = mask.unsqueeze(0).to(device)

            optimizer.zero_grad()
            output = model(img)
            loss = loss_fn(output, mask)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")
    return model

In [5]:
import time
start = time.time()
model_r34 = train_model("resnet34", samples)
r34_time = time.time() - start
r34_params = sum(p.numel() for p in model_r34.parameters())

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

In [ ]:
start = time.time()
model_eff = train_model("efficientnet-b0", samples)
eff_time = time.time() - start
eff_params = sum(p.numel() for p in model_eff.parameters())

In [ ]:
def visualize(model, sample):
    model.eval()
    img, mask = sample
    img = img.unsqueeze(0).to("cuda")
    with torch.no_grad():
        pred = model(img)[0].argmax(0).cpu().numpy()
    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    axs[0].imshow(img[0].cpu().permute(1, 2, 0))
    axs[1].imshow(mask.cpu(), cmap="gray")
    axs[2].imshow(pred, cmap="nipy_spectral")
    axs[0].set_title("Image")
    axs[1].set_title("Ground Truth")
    axs[2].set_title("Prediction")
    plt.tight_layout()
    plt.show()

visualize(model_r34, samples[0])
visualize(model_eff, samples[0])

In [ ]:
print("Comparison Summary")
print(f"ResNet34: {r34_params/1e6:.2f}M params, {r34_time:.2f}s training")
print(f"EffNet-B0: {eff_params/1e6:.2f}M params, {eff_time:.2f}s training")

ResNet34 trained slightly faster than EfficientNet-B0, taking 1.52 seconds compared to 2.03 seconds. This matches expectations since ResNet34 has a simpler architecture. However, it has a much larger size—24.45 million parameters versus EfficientNet-B0’s 6.26 million. So while EfficientNet-B0 is more compact and memory-efficient, ResNet34 may offer better segmentation accuracy due to its higher capacity and faster training.